# North DFW Housing Affordability Crisis
## Notebook 03: Feature Engineering

**Author:** Alejandro Molina  
**GitHub:** https://github.com/chooseyourtacoday  
**Date:** 2026

---

### Notebook Goal
Transform the master dataframe into analysis-ready features that directly answer our five business questions:

1. Are we heading toward a housing market correction?
2. Who is most financially at risk?
3. Is the wage gap a primary driver?
4. What does the next 2-3 years look like?
5. What interventions could realistically help?

### Features We Will Build
| Feature | Description | Answers |
|---------|-------------|----------|
| `price_to_income_ratio` | Median home value / median household income | Q2, Q3 |
| `mortgage_burden_pct` | Monthly mortgage payment as % of monthly income | Q2, Q3 |
| `home_value_yoy` | Year-over-year % change in home value | Q1 |
| `home_value_mom` | Month-over-month % change in home value | Q1 |
| `wage_growth_yoy` | Year-over-year % change in annual wages | Q3 |
| `affordability_gap` | Wage growth minus home value growth | Q3 |
| `population_growth_yoy` | Year-over-year % change in population | Q4 |
| `supply_pressure` | Permits issued relative to population | Q5 |
| `affordability_index` | Composite score (lower = less affordable) | Q1, Q2 |

## Step 1: Import Libraries & Load Master Data

In [1]:
import pandas as pd
import numpy as np
import os

CLEAN_DIR  = os.path.join('data', 'clean')
OUTPUT_DIR = os.path.join('data', 'features')
os.makedirs(OUTPUT_DIR, exist_ok=True)

master = pd.read_csv(
    os.path.join(CLEAN_DIR, 'master_dfw_housing.csv'),
    parse_dates=['date']
)

print(f'Master loaded : {master.shape}')
print(f'Date range    : {master["date"].min().date()} to {master["date"].max().date()}')
print(f'Cities        : {sorted(master["city"].unique().tolist())}')

Master loaded : (1755, 17)
Date range    : 2015-01-31 to 2026-03-31
Cities        : ['Allen', 'Celina', 'Frisco', 'McKinney', 'Plano', 'Prosper']


## Step 2: Sort Data
All time-based calculations require data sorted by zip code and date.

In [2]:
df = master.sort_values(['zip_code', 'date']).reset_index(drop=True)
print(f'Sorted shape : {df.shape}')
df.head(3)

Sorted shape : (1755, 17)


,zip_code,city,county,date,home_value,year,month,mortgage_rate,units_1unit,units_5plus,total_units,collin_avg_weekly_wage,collin_avg_annual_pay,denton_avg_weekly_wage,denton_avg_annual_pay,population,median_income
0,75002,Allen,Collin County,2015-01-31,236697.8775,2015,1,3.67,1655.0,1948.0,3603.0,1185.0,61631.0,907.0,47176.0,104627.0,118254
1,75002,Allen,Collin County,2015-02-28,239813.4993,2015,2,3.71,1598.0,1970.0,3568.0,1185.0,61631.0,907.0,47176.0,104627.0,118254
2,75002,Allen,Collin County,2015-03-31,243059.8287,2015,3,3.77,1825.0,947.0,2772.0,1185.0,61631.0,907.0,47176.0,104627.0,118254


## Step 3: Home Value Change Features
Month-over-month and year-over-year percent changes in home value.
These tell us whether the market is accelerating, slowing, or correcting.

In [3]:
# Month-over-month % change (1 month lag within each zip code)
df['home_value_mom'] = df.groupby('zip_code')['home_value'].pct_change(1) * 100

# Year-over-year % change (12 month lag within each zip code)
df['home_value_yoy'] = df.groupby('zip_code')['home_value'].pct_change(12) * 100

print('Home value change features created.')
print(f'\nMoM nulls : {df["home_value_mom"].isnull().sum()}')
print(f'YoY nulls : {df["home_value_yoy"].isnull().sum()}')

# Preview - Frisco 75034
sample = df[df['zip_code'] == 75034][['date', 'home_value', 'home_value_mom', 'home_value_yoy']].tail(6)
print(f'\nSample (Frisco 75034, last 6 months):')
print(sample.to_string(index=False))

Home value change features created.

MoM nulls : 13
YoY nulls : 156

Sample (Frisco 75034, last 6 months):
      date  home_value  home_value_mom  home_value_yoy
2025-10-31 678549.7774        0.102290       -4.425384
2025-11-30 679906.6461        0.199966       -4.261870
2025-12-31 680861.3940        0.140423       -3.968830
2026-01-31 680298.4085       -0.082687       -3.879378
2026-02-28 678663.7922       -0.240279       -3.857923
2026-03-31 676152.2558       -0.370071       -3.866011


## Step 4: Price-to-Income Ratio
Median home value divided by median household income.

**Benchmark:** A healthy market sits around 3x. Above 5x is considered severely unaffordable.
This is the single most important affordability indicator in the project.

In [4]:
df['price_to_income_ratio'] = df['home_value'] / df['median_income']

print('Price-to-income ratio created.')
print(f'\nCurrent ratio by city (most recent month):')

latest = df[df['date'] == df['date'].max()]
pti_summary = latest.groupby('city')['price_to_income_ratio'].mean().round(2).sort_values(ascending=False)
print(pti_summary.to_string())
print('\nHealthy benchmark: ~3.0x | Severely unaffordable: >5.0x')

Price-to-income ratio created.

Current ratio by city (most recent month):
city
Plano       5.37
Frisco      4.95
Allen       4.70
McKinney    4.32
Prosper     4.03
Celina      3.45

Healthy benchmark: ~3.0x | Severely unaffordable: >5.0x


## Step 5: Mortgage Burden Percentage
Estimates what percentage of monthly income goes toward a mortgage payment.

**Formula:**
- Assume 20% down payment
- Calculate monthly mortgage payment using standard amortization formula
- Divide by monthly income

**Benchmark:** Above 30% is considered cost-burdened. Above 50% is severely cost-burdened.

In [5]:
def calculate_monthly_mortgage(home_value, annual_rate, down_pct=0.20, years=30):
    """
    Calculate monthly mortgage payment using standard amortization.
    home_value  : price of the home
    annual_rate : annual interest rate as a percentage (e.g. 6.5)
    down_pct    : down payment as a decimal (default 20%)
    years       : loan term in years (default 30)
    """
    loan_amount   = home_value * (1 - down_pct)
    monthly_rate  = (annual_rate / 100) / 12
    n_payments    = years * 12

    # Standard amortization formula
    if monthly_rate == 0:
        return loan_amount / n_payments
    
    payment = loan_amount * (monthly_rate * (1 + monthly_rate) ** n_payments) / \
              ((1 + monthly_rate) ** n_payments - 1)
    return payment

# Apply to every row
df['monthly_mortgage'] = df.apply(
    lambda row: calculate_monthly_mortgage(row['home_value'], row['mortgage_rate']),
    axis=1
)

# Monthly income from annual pay
df['monthly_income'] = df['collin_avg_annual_pay'] / 12

# Mortgage burden as % of monthly income
df['mortgage_burden_pct'] = (df['monthly_mortgage'] / df['monthly_income']) * 100

print('Mortgage burden % created.')
print(f'\nCurrent mortgage burden by city (most recent month):')
burden_summary = latest.copy()
burden_summary['monthly_mortgage'] = burden_summary.apply(
    lambda row: calculate_monthly_mortgage(row['home_value'], row['mortgage_rate']), axis=1
)
burden_summary['monthly_income'] = burden_summary['collin_avg_annual_pay'] / 12
burden_summary['mortgage_burden_pct'] = (burden_summary['monthly_mortgage'] / burden_summary['monthly_income']) * 100
print(burden_summary.groupby('city')['mortgage_burden_pct'].mean().round(1).sort_values(ascending=False).to_string())
print('\nBenchmark: >30% = cost-burdened | >50% = severely cost-burdened')

Mortgage burden % created.

Current mortgage burden by city (most recent month):
city
Prosper     50.3
Frisco      44.2
Allen       37.0
Celina      35.8
Plano       35.7
McKinney    30.6

Benchmark: >30% = cost-burdened | >50% = severely cost-burdened


## Step 6: Wage Growth Year-over-Year

In [6]:
# Annual wage growth % (12 month lag)
df['wage_growth_yoy'] = df.groupby('zip_code')['collin_avg_annual_pay'].pct_change(12) * 100

print('Wage growth YoY created.')
print(f'\nAverage annual wage growth by year:')
wage_by_year = df.groupby('year')['wage_growth_yoy'].mean().round(2).dropna()
print(wage_by_year.to_string())

Wage growth YoY created.

Average annual wage growth by year:
year
2016    1.88
2017    2.13
2018    4.31
2019    2.21
2020    7.66
2021    5.60
2022    4.36
2023    2.67
2024    5.77
2025    0.00
2026    0.00


## Step 7: Affordability Gap
The difference between wage growth and home value growth.

- **Positive gap** = wages growing faster than home prices (improving affordability)
- **Negative gap** = home prices growing faster than wages (worsening affordability)

This directly answers: *Is the wage gap driving the crisis?*

In [7]:
df['affordability_gap'] = df['wage_growth_yoy'] - df['home_value_yoy']

print('Affordability gap created.')
print('\nPositive = wages outpacing homes (good)')
print('Negative = homes outpacing wages (crisis)')
print(f'\nAverage affordability gap by year:')
gap_by_year = df.groupby('year')['affordability_gap'].mean().round(2).dropna()
print(gap_by_year.to_string())

Affordability gap created.

Positive = wages outpacing homes (good)
Negative = homes outpacing wages (crisis)

Average affordability gap by year:
year
2016    -7.42
2017    -4.15
2018     1.66
2019     2.51
2020     5.20
2021   -13.15
2022   -23.53
2023     1.54
2024     3.45
2025     3.19
2026     5.71


## Step 8: Population Growth Year-over-Year

In [8]:
df['population_growth_yoy'] = df.groupby('zip_code')['population'].pct_change(12) * 100

print('Population growth YoY created.')
print(f'\nPopulation growth by city (2020-2025):')
pop_summary = df[df['year'].isin([2020, 2025])].groupby(['city', 'year'])['population'].mean().unstack()
pop_summary['growth_pct'] = ((pop_summary[2025] - pop_summary[2020]) / pop_summary[2020] * 100).round(1)
print(pop_summary.to_string())

Population growth YoY created.

Population growth by city (2020-2025):
year          2020      2025  growth_pct
city                                    
Allen     104627.0  112748.0         7.8
Celina     16739.0   47908.0       186.2
Frisco    200509.0  240561.0        20.0
McKinney  195308.0  229734.0        17.6
Plano     285494.0  292788.0         2.6
Prosper    30174.0   44586.0        47.8


## Step 9: Supply Pressure Index
New housing permits relative to population.
Low supply relative to population growth = upward price pressure.

In [9]:
# Permits per 1000 residents
df['supply_pressure'] = (df['total_units'] / df['population']) * 1000

print('Supply pressure index created.')
print(f'\nAverage supply pressure by year:')
supply_by_year = df.groupby('year')['supply_pressure'].mean().round(3).dropna()
print(supply_by_year.to_string())

Supply pressure index created.

Average supply pressure by year:
year
2015    45.147
2016    40.616
2017    46.183
2018    46.062
2019    42.042
2020    41.385
2021    46.548
2022    39.938
2023    32.381
2024    31.375
2025    29.402
2026    21.657


## Step 10: Composite Affordability Index
A single score combining price-to-income ratio, mortgage burden, and affordability gap.
We normalize each component to a 0-1 scale then average them.

**Lower score = less affordable.**

In [10]:
def normalize(series):
    """Normalize a series to 0-1 scale."""
    return (series - series.min()) / (series.max() - series.min())

# Higher PTI = less affordable, so we invert it
df['pti_norm']     = 1 - normalize(df['price_to_income_ratio'])

# Higher burden % = less affordable, so we invert it
df['burden_norm']  = 1 - normalize(df['mortgage_burden_pct'])

# Higher gap = more affordable (wages outpacing homes)
df['gap_norm']     = normalize(df['affordability_gap'])

# Composite affordability index (average of three components)
df['affordability_index'] = df[['pti_norm', 'burden_norm', 'gap_norm']].mean(axis=1)

print('Affordability index created.')
print('\nCurrent affordability index by city (lower = less affordable):')
ai_latest = df[df['date'] == df['date'].max()]
print(ai_latest.groupby('city')['affordability_index'].mean().round(3).sort_values().to_string())

Affordability index created.

Current affordability index by city (lower = less affordable):
city
Frisco      0.558
Prosper     0.585
Plano       0.589
Allen       0.629
McKinney    0.705
Celina      0.745


## Step 11: Feature Summary & Null Check

In [11]:
new_features = [
    'home_value_mom', 'home_value_yoy', 'price_to_income_ratio',
    'monthly_mortgage', 'monthly_income', 'mortgage_burden_pct',
    'wage_growth_yoy', 'affordability_gap', 'population_growth_yoy',
    'supply_pressure', 'affordability_index'
]

print('=== NEW FEATURES NULL CHECK ===')
print(df[new_features].isnull().sum())

print(f'\n=== FEATURE SUMMARY ===')
print(df[new_features].describe().round(3))

=== NEW FEATURES NULL CHECK ===
home_value_mom            13
home_value_yoy           156
price_to_income_ratio      0
monthly_mortgage           0
monthly_income             0
mortgage_burden_pct        0
wage_growth_yoy          156
affordability_gap        156
population_growth_yoy    156
supply_pressure            0
affordability_index        0
dtype: int64

=== FEATURE SUMMARY ===
       home_value_mom  home_value_yoy  price_to_income_ratio  \
count        1742.000        1599.000               1755.000   
mean            0.472           6.426                  3.802   
std             0.947          10.016                  1.102   
min            -1.886          -9.128                  1.785   
25%            -0.100           0.163                  2.966   
50%             0.271           3.265                  3.665   
75%             0.783           9.212                  4.499   
max             4.153          42.522                  6.737   

       monthly_mortgage  monthly_i

## Step 12: Save Feature Dataset

In [12]:
# Drop intermediate normalization columns
df = df.drop(columns=['pti_norm', 'burden_norm', 'gap_norm'])

output_path = os.path.join(OUTPUT_DIR, 'dfw_housing_features.csv')
df.to_csv(output_path, index=False)

print('=' * 60)
print('NOTEBOOK 03 COMPLETE — FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'\nFeature dataset saved to : {output_path}')
print(f'Rows                     : {df.shape[0]}')
print(f'Columns                  : {df.shape[1]}')
print(f'\nNew features built       : {len(new_features)}')
for f in new_features:
    print(f'  {f}')
print('\nNext: Notebook 04 — Predictive Model')

NOTEBOOK 03 COMPLETE — FEATURE ENGINEERING SUMMARY

Feature dataset saved to : data\features\dfw_housing_features.csv
Rows                     : 1755
Columns                  : 28

New features built       : 11
  home_value_mom
  home_value_yoy
  price_to_income_ratio
  monthly_mortgage
  monthly_income
  mortgage_burden_pct
  wage_growth_yoy
  affordability_gap
  population_growth_yoy
  supply_pressure
  affordability_index

Next: Notebook 04 — Predictive Model
